# Task 4 — Place Each Item in the Right Stage

**Ali Raza** · Roll No: **su92-bsdsm-f23-018** · Section: **BSDS-7A**
Data Warehouse & BI — Week 1

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#9aa4b2", "axes.labelcolor": "#1f2937",
    "text.color": "#1f2937", "xtick.color": "#4b5563", "ytick.color": "#4b5563",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e5e7eb", "grid.linewidth": 0.7,
    "axes.axisbelow": True, "font.size": 10,
})
INK, BLUE, RED, GREY = "#1f2937", "#2563eb", "#dc2626", "#9ca3af"
os.makedirs("charts", exist_ok=True)
NOTE = "DWBI Week 1 · synthetic course data"


## 1. The eight items — two per stage

| | Item | Stage | One-line justification |
|---|---|---|---|
| A | Removing duplicate orders | **ETL / Staging** | cleaning happens on the way in, once, before anything downstream sees the data |
| B | CEO dashboard on a phone | **BI Applications** | it *presents* information to a decision-maker; it stores nothing |
| C | Branch POS billing system | **Source Systems** | where the business happens — the origin of the sales facts |
| D | Sales star schema by date, branch, product | **Presentation Area** | the clean, conformed model analysts query — the serving counter |
| E | HR team's staff Excel sheet | **Source Systems** | it originates facts (headcount) nothing else holds — *if* it's governed (see below) |
| F | Converting `CL-150` to `Cola 1.5L` | **ETL / Staging** | conforming aliases to one canonical value is the kitchen's job |
| G | Weekly PDF report emailed to managers | **BI Applications** | a finished information product delivered to readers |
| H | Finance data mart | **Presentation Area** | a subject-oriented slice of the conformed model, built for querying |

## 2. My own diagram — drawn, not screenshotted

In [2]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
fig, ax = plt.subplots(figsize=(13, 5.2))
ax.set_xlim(0, 13); ax.set_ylim(0, 5.2); ax.axis("off")
stages = [("SOURCE\nSYSTEMS", 1.5, "#dbeafe", "#1d4ed8"),
          ("ETL /\nSTAGING", 4.7, "#fef3c7", "#b45309"),
          ("PRESENTATION\nAREA", 7.9, "#dcfce7", "#15803d"),
          ("BI\nAPPLICATIONS", 11.1, "#fae8ff", "#a21caf")]
items = {0: ["C · Branch POS", "E · HR Excel*"],
         1: ["A · Deduplicate", "F · CL-150 → Cola"],
         2: ["D · Star schema", "H · Finance mart"],
         3: ["B · CEO dashboard", "G · Weekly PDF"]}
for (title, x, bg, fg), it in zip(stages, items.values()):
    box = FancyBboxPatch((x-1.35, 0.6), 2.7, 3.6, boxstyle="round,pad=0.12",
                         facecolor=bg, edgecolor=fg, lw=2)
    ax.add_patch(box)
    ax.text(x, 3.75, title.replace("\\n", "\n"), ha="center", va="center",
            fontsize=12, weight="bold", color=fg)
    for i, t in enumerate(it):
        ax.text(x, 2.9 - i*0.75, t, ha="center", va="center", fontsize=9.5,
                color="#1f2937", bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="#cbd5e1"))
for i in range(3):
    a = FancyArrowPatch((stages[i][1]+1.35, 2.4), (stages[i+1][1]-1.35, 2.4),
                        arrowstyle="-|>", mutation_scale=18, color="#475569", lw=2)
    ax.add_patch(a)
ax.text(6.5, 0.18, "* HR Excel only if governed — see the question below", ha="center",
        fontsize=9, style="italic", color="#64748b")
ax.text(6.5, 4.85, "The DW/BI pipeline — my own drawing (data flows left → right)",
        ha="center", fontsize=12, weight="bold", color=INK)
fig.savefig("charts/t4_architecture.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("saved charts/t4_architecture.png")

saved charts/t4_architecture.png


![My architecture diagram](charts/t4_architecture.png)

## 3. Extending the restaurant metaphor

- **A diner wandering into the kitchen to grab food off the stove** = users querying the **source systems** directly. That's the CEO's report on the live checkout database from Task 2: the kitchen is mid-service, and the diner slows down every cook. In a real warehouse this is the analytics query that locks the POS tables at 6pm on a Friday.
- **The kitchen serving a dish without tasting it** = loading data with **no validation**. That's order `1008`'s Rs 3,600 sailing straight into the Beverages total — one taste (amount vs price master) would have caught it, and now the CEO is eating it.
- **Two kitchens cooking the same dish to different recipes** = **two ETL pipelines / no single staging area**. That's the three departments' three "sales" figures from Task 3: same ingredients, different recipes, three different dishes all called "sales last month".

## 4. Three of Kimball's goals — and their Superstore violations

1. **"The DW/BI system must present information consistently."** Violated by `Beverages` / `Bev.` / blank and `Cola 1.5L` / `CL-150` — the same thing called three names means no two reports agree.
2. **"It must be a single, authoritative foundation for decision-making."** Violated by three departments keeping three private "sales" definitions (Task 3) — there is no one number the CEO can trust.
3. **"It must be easy to access and understand."** Violated by cryptic source codes (`CL-150`, `OJ-1L`) exposed raw — a manager shouldn't need a decoder ring to read a sales report.

## The question to answer — the HR Excel sheet

**The case for:** the data exists nowhere else — headcount by department lives in that sheet, and excluding it means every "sales per employee" analysis is impossible. A source is defined by *origination*, not by technology.

**The case against:** it's one person's laptop: no audit trail, no access control, silent overwrites (exactly the Ali problem from Task 3 — someone edits a row and history is gone), and no schema contract.

**My choice:** accept it as a source system — but **only** with requirements: a named owner, frozen versioned drops (never the live file), a declared schema, and loading through staging with validation like every other source. Trust is earned by process, not by file format.